In [ ]:
import random
from datetime import datetime, timedelta

from faker import Faker
from great_expectations.checkpoint import Checkpoint
from great_expectations.exceptions import DataContextError
from pyspark.sql import SparkSession
from pyspark.sql.functions import lit, when, col
from pyspark.sql.types import StructType, StructField, LongType, TimestampType, StringType

import great_expectations as gx

In [ ]:
schema = StructType([
    StructField("id", LongType(), False),
    StructField("published", TimestampType(), True),
    StructField("subject", StringType(), True),
    StructField("keyword", StringType(), True),
    StructField("title", StringType(), True),
    StructField("summary", StringType(), True),
    StructField("description", StringType(), True),
    StructField("original_link", StringType(), True),
    StructField("link", StringType(), True),
    StructField("created_at", TimestampType(), True),
    StructField("updated_at", TimestampType(), True),
])

In [ ]:
def random_dt(within_days=30):
    return datetime.now() - timedelta(days=random.randint(0, within_days), hours=random.randint(0, 23), minutes=random.randint(0, 59), seconds=random.randint(0, 59), )


def generate_news_data(fake: Faker, n: int, start_id: int = 1):
    return [generate_news_row(fake, start_id + i) for i in range(n)]


def generate_news_row(fake: Faker, id_: int):
    published = random_dt(14)
    return {
        "id": id_,
        "published": published,
        "subject": random.choice(["경제", "사회", "정치", "IT", "국제", "문화"]),
        "keyword": ", ".join(fake.words(nb=random.randint(2, 6)))[:200],
        "title": fake.sentence(nb_words=12)[:1000],
        "summary": fake.text(max_nb_chars=600)[:4000],
        "description": fake.text(max_nb_chars=2500),
        "original_link": fake.url()[:500],
        "link": fake.url()[:500],
        "created_at": published,
        "updated_at": published,
    }

In [ ]:
spark = SparkSession.builder.appName("Example Great Expectations").master("spark://spark-master.mmix.io:7077").config("spark.sql.shuffle.partitions", "1").getOrCreate()
news = spark.createDataFrame(generate_news_data(Faker("ko_KR"), 10), schema=schema)
news_bad = news.withColumn("title", when(col("id") % 200 == 0, lit(None)).otherwise(col("title"))).withColumn("link", when(col("id") % 333 == 0, lit("not-a-url")).otherwise(col("link")))

In [ ]:
context = gx.get_context(mode="file", context_root_dir="/Users/genius/Workspace/enjoy-workreduce/notebooks/spark/great_expectations")
#context = gx.get_context()

In [ ]:
datasource_name = "news_datasource"
asset_name = "news_datasource_asset"
batch_definition_name = "news_datasource_batch_definition_whole_dataframe"
suite_name = "news_datasource_suite"
validation_definition_name = "news_datasource_validation_definition"
checkpoint_name = "news_datasource_runtime_checkpoint"

In [ ]:
news_datasource = (
    context.data_sources.get(datasource_name)
    if datasource_name in context.data_sources.all()
    else context.data_sources.add_spark(name=datasource_name)
)

df_asset = (
    news_datasource.get_asset(asset_name)
    if asset_name in news_datasource.get_asset_names()
    else news_datasource.add_dataframe_asset(name=asset_name)
)

try:
    batch_def = df_asset.get_batch_definition(batch_definition_name)
except KeyError:
    batch_def = df_asset.add_batch_definition_whole_dataframe(name=batch_definition_name)

batch_parameters = {"dataframe": news_bad}

try:
    suite = context.suites.get(suite_name)
except DataContextError:
    suite = context.suites.add(gx.ExpectationSuite(name=suite_name))

suite.expectations.clear()
required_columns = ["id", "published", "subject", "keyword", "title", "summary", "description", "original_link", "link", "created_at", "updated_at"]
allowed_subjects = ["경제", "사회", "정치", "IT", "국제", "문화"]
url_regex = r"^https?://.+"

for c in required_columns:
    suite.add_expectation(gx.expectations.ExpectColumnToExist(column=c))

suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="id"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="title", mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="created_at", mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="updated_at", mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="link", mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="original_link", mostly=0.99))

suite.add_expectation(gx.expectations.ExpectColumnValuesToBeUnique(column="id"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="id", min_value=1))

suite.add_expectation(gx.expectations.ExpectColumnValuesToBeInSet(column="subject", value_set=allowed_subjects, mostly=0.98))

suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToBeBetween(column="keyword", min_value=0, max_value=200, mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToBeBetween(column="title", min_value=5, max_value=1000, mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToBeBetween(column="summary", min_value=0, max_value=4000, mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToBeBetween(column="description", min_value=0, max_value=2600, mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToBeBetween(column="link", min_value=0, max_value=500, mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValueLengthsToBeBetween(column="original_link", min_value=0, max_value=500, mostly=0.99))

suite.add_expectation(gx.expectations.ExpectColumnValuesToMatchRegex(column="original_link", regex=url_regex, mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnValuesToMatchRegex(column="link", regex=url_regex, mostly=0.99))

suite.add_expectation(gx.expectations.ExpectColumnPairValuesAToBeGreaterThanB(column_A="updated_at", column_B="created_at", or_equal=True, mostly=0.99))

suite.add_expectation(gx.expectations.ExpectColumnPairValuesToBeEqual(column_A="created_at", column_B="published", mostly=0.99))
suite.add_expectation(gx.expectations.ExpectColumnPairValuesToBeEqual(column_A="updated_at", column_B="published", mostly=0.99, ))
suite.save()

try:
    validation_definition = context.validation_definitions.get(validation_definition_name)
except DataContextError:
    validation_definition = context.validation_definitions.add(gx.ValidationDefinition(name=validation_definition_name, data=batch_def, suite=suite))

validation_results = validation_definition.run(batch_parameters=batch_parameters, result_format={"result_format": "COMPLETE"})

In [ ]:
actions = [{"name": "update_data_docs", "type": "update_data_docs", "site_names": ["minio_site"]}]
try:
    checkpoint = context.checkpoints.get(checkpoint_name)
except DataContextError:
    checkpoint = Checkpoint(name=checkpoint_name, validation_definitions=[{"name": validation_definition_name}], actions=actions)
    checkpoint = context.checkpoints.add(checkpoint)
    minio_store = context.stores["validation_results_store_minio"]

result = checkpoint.run(batch_parameters=batch_parameters)

In [ ]:
spark.stop()